# UC-PROC-3 — Asset-Scoped Processing via OGC API - Processes

**Persona:** Data Engineer

**Goal:** After uploading an asset, list the OGC API - Processes that are runnable
against that asset, execute the canonical asset process (`gdal`) on it, poll the job
to completion, retrieve the result, and dismiss the job.

**Background:** Asset-scoped work runs as a plain OGC process mounted at the asset URL —
there is no bespoke per-asset task dispatcher. The `asset_id` (and `catalog_id` /
`collection_id`) are taken from the URL path and injected into the process `inputs`
automatically, so the request body carries only the process-specific parameters. The
`gdal` process inspects the asset with GDAL/OGR and enriches its metadata; it is the
canonical, registered `ASSET`-scoped process.

**Asset-scoped routes (router prefix `/processes`):**
```
GET  /processes/catalogs/{c}[/collections/{col}]/assets/{a}/processes
POST /processes/catalogs/{c}[/collections/{col}]/assets/{a}/processes/{id}/execution   (Prefer: respond-sync|respond-async)
```

**Prerequisites:**
- Asset already uploaded via the assets notebook (`assets/01`) — `ASSET_ID` must be known
- The `gdal` process available (requires the GDAL/osgeo runtime, in-process or on a worker)
- Write-capable JWT in `DYNASTORE_WRITE_TOKEN`

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_WRITE_TOKEN`, `CATALOG_ID`,
`COLLECTION_ID`, `ASSET_ID`  
**Optional:** `PROCESS_ID` (defaults to `gdal`)

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
WRITE_TOKEN = os.environ.get("DYNASTORE_WRITE_TOKEN", "")
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")
ASSET_ID = os.environ.get("ASSET_ID", "")
# Canonical ASSET-scoped process; override to exercise a different asset process.
PROCESS_ID = os.environ.get("PROCESS_ID", "gdal")

headers = {
    "Authorization": f"Bearer {WRITE_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

# Asset-scoped OGC routes, parameterised by the env above. The asset URL can be
# catalog-level or collection-level; both mount the same asset-scoped processes.
ASSET_BASE = f"/processes/catalogs/{CATALOG_ID}/assets/{ASSET_ID}"

job_id = None
PROCESS_AVAILABLE = False

_SKIP = not ASSET_ID
if _SKIP:
    print("SKIP: ASSET_ID not set. Upload an asset via assets/01 and set ASSET_ID env var.")

print(f"Connected to {BASE_URL}")
print(f"CATALOG_ID={CATALOG_ID}  COLLECTION_ID={COLLECTION_ID}  ASSET_ID={ASSET_ID or '(none)'}  SKIP={_SKIP}")
print(f"PROCESS_ID={PROCESS_ID}")

In [ ]:
if _SKIP:
    print("SKIP: no ASSET_ID — skipping process listing.")
else:
    # List the OGC processes runnable against this asset. Only ASSET-scoped
    # processes are advertised at the asset mount.
    r = client.get(f"{ASSET_BASE}/processes")
    print(r.status_code)
    assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

    body = r.json()
    processes = body.get("processes", body) if isinstance(body, dict) else body
    if isinstance(processes, dict):
        processes = list(processes.values())

    print(f"Asset-scoped processes ({len(processes)}):")
    for proc in processes:
        proc_id = proc.get("id", proc.get("name", "?"))
        title = proc.get("title", proc.get("summary", ""))
        print(f"  {proc_id}: {title}")

    # Confirm the target process is registered for this asset. If `gdal` is not
    # available (no GDAL runtime in-process and no worker carrying it), fall back
    # to the first listed process so the rest of the walkthrough still runs.
    available_ids = [p.get("id") for p in processes if p.get("id")]
    if PROCESS_ID not in available_ids and available_ids:
        print(f"NOTE: PROCESS_ID '{PROCESS_ID}' not registered here; falling back to '{available_ids[0]}'")
        PROCESS_ID = available_ids[0]
    PROCESS_AVAILABLE = PROCESS_ID in available_ids
    print(f"\nSelected PROCESS_ID={PROCESS_ID}  (available={PROCESS_AVAILABLE})")

In [ ]:
if _SKIP or not PROCESS_AVAILABLE:
    print("SKIP: no ASSET_ID / process unavailable — skipping execution.")
else:
    # Execute the process on this asset, asynchronously. The asset_id, catalog_id
    # (and collection_id at a collection mount) are injected from the URL path, so
    # the body carries only process-specific parameters — do NOT repeat the path
    # identifiers in `inputs` (a conflicting value would be rejected with 400).
    execution_payload = {
        "inputs": {},
        "outputs": {"info": {"transmissionMode": "value"}},
    }

    r = client.post(
        f"{ASSET_BASE}/processes/{PROCESS_ID}/execution",
        content=json.dumps(execution_payload),
        headers={"Prefer": "respond-async"},
    )
    print(r.status_code, r.text[:400])
    # Async execution creates a job: 201 Created + Location header pointing at the
    # job status URL (a fast process under respond-sync would instead return 200).
    assert r.status_code in (200, 201), (
        f"Expected 201 (job created) or 200 (sync result), got {r.status_code}: {r.text}"
    )

    receipt = r.json()
    if r.status_code == 201:
        job_id = receipt.get("jobID", receipt.get("job_id", ""))
        assert job_id, f"No jobID in async execution response: {receipt}"
        print(f"Process accepted: job_id={job_id}  (Location: {r.headers.get('Location')})")
    else:
        print("Synchronous result returned; no job to poll.")
        print(json.dumps(receipt, indent=2)[:800])

In [ ]:
if _SKIP or not job_id:
    print("SKIP: no job to poll.")
else:
    # Poll the job to a terminal status. OGC job states map from the queue:
    # accepted (PENDING) -> running (ACTIVE) -> successful / failed / dismissed.
    # `gdal` runs on whichever service carries the GDAL runtime: in-process where
    # available, otherwise routed to the worker queue. On a stack with no worker
    # the job can legitimately stay `accepted`; treat that as inconclusive rather
    # than a hard failure so the walkthrough still demonstrates the lifecycle.
    job_status = "accepted"
    for attempt in range(20):
        resp = client.get(f"/processes/jobs/{job_id}")
        assert resp.status_code == 200, (
            f"Expected 200 polling job, got {resp.status_code}: {resp.text}"
        )
        job_status = resp.json().get("status", "")
        print(f"Attempt {attempt + 1}: status={job_status}")
        if job_status in ("successful", "failed", "dismissed"):
            break
        time.sleep(min(2 ** attempt, 30))

    assert job_status != "failed", f"Process job failed: {resp.text}"
    if job_status == "successful":
        print(f"Process completed: status={job_status}")
    else:
        print(
            f"Process job not yet terminal (status='{job_status}'). No executor "
            f"advanced it — ensure a service carries the GDAL runtime for '{PROCESS_ID}'."
        )

In [ ]:
if _SKIP or not job_id or job_status != "successful":
    print("SKIP: no successfully-completed job to fetch results for.")
else:
    # Retrieve the process result (the gdal/ogr info computed for the asset).
    r = client.get(f"/processes/jobs/{job_id}/results")
    print(r.status_code)
    assert r.status_code == 200, f"Expected 200 fetching results, got {r.status_code}: {r.text}"

    report = r.json()
    print("Process result:")
    print(json.dumps(report, indent=2)[:1200])

In [ ]:
if _SKIP or not job_id:
    print("SKIP: no job to dismiss.")
else:
    # Dismiss the job (OGC Part 1 §13: DELETE /jobs/{id}).
    r = client.delete(f"/processes/jobs/{job_id}")
    print(r.status_code)
    assert r.status_code in (200, 204), (
        f"Expected 200 or 204 dismissing job, got {r.status_code}: {r.text}"
    )
    print(f"Job {job_id} dismissed.")

## Edge Cases

In [ ]:
# Edge case 1: unknown asset -> 404
# Listing processes for an asset that does not exist in CATALOG_ID returns 404.
# The platform does not leak cross-tenant asset existence via the error shape.
fake_asset_id = "00000000-0000-0000-0000-000000000000"
r = client.get(f"/processes/catalogs/{CATALOG_ID}/assets/{fake_asset_id}/processes")
print(r.status_code)
assert r.status_code in (200, 404), (
    f"Expected 404 (unknown asset) or 200 (empty list), got {r.status_code}: {r.text}"
)
if r.status_code == 404:
    print("404 confirmed: non-existent asset correctly returns 404.")
else:
    print("200: this deployment lists asset processes statically (no asset existence probe).")

In [ ]:
if _SKIP:
    print("SKIP: no ASSET_ID — skipping process-mismatch edge case.")
else:
    # Edge case 2: process not runnable at the asset mount.
    # Two distinct rejections, both *before* any job row is written:
    #   * unknown process id              -> 404 Not Found
    #   * real process with a non-ASSET   -> 400 Bad Request (scope/URL mismatch)
    #     scope (e.g. a CATALOG process)
    # Here we use a clearly non-existent id; the platform returns 404 (or 400 if
    # the id collides with a registered process declaring an incompatible scope).
    bogus_process_id = "this-process-does-not-exist"
    r = client.post(
        f"{ASSET_BASE}/processes/{bogus_process_id}/execution",
        content=json.dumps({"inputs": {}}),
        headers={"Prefer": "respond-async"},
    )
    print(r.status_code)
    assert r.status_code in (400, 404), (
        f"Expected 404 (unknown process) or 400 (scope mismatch), "
        f"got {r.status_code}: {r.text}"
    )
    print(
        f"{r.status_code} confirmed: "
        f"{'process not registered' if r.status_code == 404 else 'process not executable at the asset mount'}."
    )

In [ ]:
# Edge case 3: collection-scoped asset mount
# The same asset-scoped processes are reachable at the longer collection-level
# asset URL. The platform uses the COLLECTION_ID segment to resolve the correct
# schema and apply collection-level write policies before executing.
#
#   GET  /processes/catalogs/{c}/collections/{col}/assets/{a}/processes
#   POST /processes/catalogs/{c}/collections/{col}/assets/{a}/processes/{id}/execution
#
# Verify the longer form lists the same processes as the shorter catalog-level form:
if not _SKIP:
    coll_base = (
        f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
        f"/assets/{ASSET_ID}"
    )
    r_coll = client.get(f"{coll_base}/processes")
    print(r_coll.status_code)
    assert r_coll.status_code in (200, 404), (
        f"Expected 200 or 404 for collection-scoped process list, "
        f"got {r_coll.status_code}: {r_coll.text}"
    )
    if r_coll.status_code == 200:
        coll_body = r_coll.json()
        coll_list = (
            coll_body.get("processes", coll_body)
            if isinstance(coll_body, dict) else coll_body
        )
        if isinstance(coll_list, dict):
            coll_list = list(coll_list.values())
        coll_ids = [p.get("id", "") for p in coll_list]
        print(f"Collection-scoped process ids: {coll_ids}")
        if PROCESS_AVAILABLE:
            assert PROCESS_ID in coll_ids, (
                f"PROCESS_ID '{PROCESS_ID}' not present in collection-scoped listing: {coll_ids}"
            )
            print("Collection-scoped URL form confirmed equivalent.")
    else:
        print(
            "404: collection-scoped asset route not mounted in this deployment.\n"
            f"Use the catalog-level form: GET {ASSET_BASE}/processes"
        )
else:
    print("SKIP: no ASSET_ID — skipping collection-scoped variant check.")

client.close()